# Custom steering vectors · Show-o2 1.5B

Build one direction from a contrastive text pair using the understanding backbone.
Use an A100 or L4 GPU and High-RAM. All text, including right-side space padding,
is visible to causal attention. This notebook builds vectors only; application to
understanding answers or image generation is a later experiment.

## 1. Set up a fresh Colab runtime

Clone this branch and pin the original model implementation. Restart the session
after installation, then continue with section 2.

In [ ]:
!nvidia-smi
!git clone --branch experiment/custom-steering-vectors --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1


## 2. Load the shared adapter

Run once after the restart. Later cells reuse this adapter.

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from IPython.display import Markdown, display
from google.colab import files

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter
from experiments.custom_utils import (
    prepare_pair, capture_pair, build_steering_vector, make_steering_vector, estimate_direction,
)

torch.manual_seed(42)
adapter = ShowO2Adapter()
OUTPUT = PROJECT / "outputs/custom-steering"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(MODEL_ID, torch.cuda.get_device_name(0), adapter.dtype)
print("Decoder blocks:", len(adapter.layers))

## 3. Define and inspect one text pair

Raw text receives one shared BOS, with no chat header, answer suffix, or EOS.
`right_pad` appends ordinary space tokens to the shorter sequence, following the
ActAdd alignment convention. It does not shift the text to the right.

Token positions include BOS. With padding enabled, `"last"` means the final
**padded** position, which may be a space. Inspect the IDs before selecting tokens.

In [ ]:
pair_pos = "A red car parked on a street."
pair_neg = "A blue car parked on a street."
alignment = "right_pad"

pair = prepare_pair(adapter, pair_pos, pair_neg, alignment=alignment)
for side, ids in pair["token_ids"].items():
    print("\n", side, "| added spaces:", pair["padding"][side])
    for position, token_id in enumerate(ids):
        print(position, token_id, repr(adapter.tokenizer.decode([token_id])))

## 4. Capture selected decoder layers

Use `layers=0`, `layers=[0, 1, 12]`, `layers=slice(0, 8)`, or `layers="all"`.
Indices are zero-based; `-1` selects the last block. Captures are block outputs
before final normalization, stored on CPU in FP32.

Both sides run independently with causal masks and frozen weights. No text is
generated. Capture once, then reuse these tensors for different reductions.

In [ ]:
layers = "all"
captured = capture_pair(adapter, pair, layers=layers)

for layer in captured["layers"]:
    print(layer, {
        side: tuple(captured[side][layer].shape)
        for side in ("positive", "negative")
    })

## 5. Select tokens and construct the direction

Selections: `0` = first token, `1` = second, `"last"` = final position,
`[1, 3]` = selected positions, `slice(1, None)` = all except BOS, or `"all"`.
Negative positions count from each sequence's end.

For each layer: select states, pool tokens separately for each example, then
estimate a direction. The default method subtracts the two pooled states. `"none"` keeps a tokenwise matrix; `"sum"` sums token states;
`"mean"` and `"average"` are identical. Unequal selected lengths require alignment.

Layer reduction is separate and optional. `vectors[layer]` always retains each
layer's result; `combined` is populated only when layer reduction is requested.
No normalization or steering coefficient is applied.
The preview shows six coordinates of the first selected token at the first selected
layer: `h_positive - h_negative`. It displays real captured values, not toy data.


In [ ]:
tokens = "last"
token_reduction = "mean"
layer_reduction = "none"
direction_method = "difference"

result = build_steering_vector(
    captured,
    tokens=tokens,
    token_reduction=token_reduction,
    layer_reduction=layer_reduction,
    direction_method=direction_method,
)
for layer, vector in result["vectors"].items():
    print(layer, tuple(vector.shape), "norm:", vector.norm().item())
print("Selected positions:", result["metadata"]["selected_positions"])
print("Combined:", result["combined"])

preview_layer = captured["layers"][0]
h_pos = result["positive"][preview_layer][0, :6]
h_neg = result["negative"][preview_layer][0, :6]
print("Preview layer:", preview_layer)
print("Positive token position:", result["metadata"]["selected_positions"]["positive"][0])
print("Negative token position:", result["metadata"]["selected_positions"]["negative"][0])
rows = zip(h_pos.tolist(), h_neg.tolist(), (h_pos - h_neg).tolist())
display(Markdown(
    "| Coordinate | h+ | h- | h+ - h- |\n| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {a:.6f} | {b:.6f} | {d:.6f} |\n"
              for i, (a, b, d) in enumerate(rows))
))


## 6. Compare reductions without another model forward

These operations aggregate token differences from this single pair; they do not
estimate a mean across a dataset. Averaging layers mixes residual depths and is an
explicit experimental choice, not a replacement for layer-specific vectors.
The table shows `sum(h+ - h-) / number_of_tokens` next to the calculated token
mean direction, using six coordinates from one layer. The BOS is included.


In [ ]:
mean_result = build_steering_vector(
    captured, tokens="all", token_reduction="mean", layer_reduction="none",
)
sum_result = build_steering_vector(
    captured, tokens="all", token_reduction="sum", layer_reduction="none",
)
layer_mean_result = build_steering_vector(
    captured, tokens="last", token_reduction="mean", layer_reduction="mean",
)

layer = captured["layers"][0]
print("Token mean:", mean_result["vectors"][layer][:12])
print("Token sum:", sum_result["vectors"][layer][:12])
print("Layer mean:", layer_mean_result["combined"][:12])

n_tokens = mean_result["differences"][layer].shape[0]
sums = sum_result["vectors"][layer][:6]
means = mean_result["vectors"][layer][:6]
print("Preview layer:", layer, "| selected tokens:", n_tokens)
rows = zip(sums.tolist(), (sums / n_tokens).tolist(), means.tolist())
display(Markdown(
    "| Coordinate | Sum of differences | Sum / token count | Mean direction |\n"
    "| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {s:.6f} | {q:.6f} | {m:.6f} |\n"
              for i, (s, q, m) in enumerate(rows))
))


## 7. Stacked hidden states and a custom direction method

`direction_method="svd"` stacks the pooled positive and negative hidden states
without subtracting their mean. It returns the leading right singular vector with
unit norm, oriented toward the positive-minus-negative contrast when possible.
This is uncentered SVD, sometimes called uncentered PCA. It can capture structure
shared by both examples rather than the distinction between them.

A direction callable receives a `[examples, hidden]` matrix of pooled states:
positive examples first, then negative examples. It returns one `[hidden]` vector.
Centering belongs inside this callable when the experiment calls for it.

`estimate_direction(positive, negative, method)` also accepts matrices with many
examples per side. The current capture pipeline still builds one pair. Future
20/50/100-pair experiments can stack their pooled states and reuse this estimator.
Layers stay separate; token positions are not treated as independent examples.


In [ ]:
svd_result = build_steering_vector(
    captured, tokens="all", token_reduction="mean",
    direction_method="svd", layer_reduction="none",
)
print("Uncentered direction:", svd_result["vectors"][captured["layers"][0]][:12])

def centered_pca(states):
    centered = states - states.mean(dim=0, keepdim=True)
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    return vh[0]

custom_result = build_steering_vector(
    captured, tokens="all", token_reduction="mean",
    direction_method=centered_pca, layer_reduction="none",
)
print("Custom centered PCA:", custom_result["vectors"][captured["layers"][0]][:12])


## 8. Use the same pipeline through one function

This cell performs two new forwards. Use the stepwise functions above when
inspecting intermediate results or changing only aggregation settings.

In [ ]:
single_call = make_steering_vector(
    adapter,
    pair_pos=pair_pos,
    pair_neg=pair_neg,
    alignment="right_pad",
    tokens="last",
    layers=[0, 1, 12],
    token_reduction="mean",
    direction_method="difference",
    layer_reduction="none",
)
print({layer: tuple(vector.shape) for layer, vector in single_call["vectors"].items()})

## 9. Save and download

Save the result chosen in section 5, full captures, exact token IDs, and revisions.
Custom reducer names are recorded; keep their definitions with the notebook.
Model weights use the adapter's default Hugging Face revisions.

In [ ]:
result["metadata"].update(
    model=MODEL_ID,
    seed=42,
    dtype=str(adapter.dtype),
    project_revision=get_ipython().getoutput("git -C /content/show-o2-space rev-parse HEAD")[0],
    upstream_revision=get_ipython().getoutput("git -C /content/Show-o rev-parse HEAD")[0],
)
torch.save(result, OUTPUT / "steering_vector.pt")
torch.save(captured, OUTPUT / "captures.pt")
(OUTPUT / "metadata.json").write_text(
    json.dumps(result["metadata"], indent=2), encoding="utf-8",
)
archive = shutil.make_archive("/content/custom-steering-results", "zip", OUTPUT)
files.download(archive)

## References

- [ActAdd alignment](https://github.com/ajobic/activation_additions)
- [PyTorch forward hooks](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)
- [CAST extraction](https://arxiv.org/html/2409.05907v1#S3.SS3)
- [PyTorch SVD](https://docs.pytorch.org/docs/stable/generated/torch.linalg.svd.html)
- [Show-o2 implementation](https://github.com/showlab/Show-o/tree/45a5a2de01d1ebd10cd5864d29310a76476cdf23/show-o2)

Local numerical checks use synthetic states and a small decoder. Full Show-o2
extraction in Colab remains to be verified. Push this branch before opening its
Colab link.